In [ ]:
LiH_results = {"Qiskit-PauliEvolutionDefault":216,
              "Qiskit-Rustiq-greedy-preserve":103,
              "Qiskit-Rustiq-greedy-no_preserve": 56 
              "QuCLEAR-transpiled": 74,
               "MonteQ-greedy-whole": 70,
              "MonteQ-greedy-sets-div": 95,
              "Tket-whole": 132}

H2O_results = {"Qiskit-PauliEvolutionDefault":1024,
              "Qiskit-Rustiq-greedy-preserve":337,
               "Qiskit-Rustiq-greedy-no_preserve":271,
              "QuCLEAR-transpiled": 274,
              "MonteQ-greedy-whole": 237,
              "MonteQ-greedy-sets-div": 314,
              "Tket-whole": 505}

Benzene_results = {"Qiskit-PauliEvolutionDefault":9880,
              "Qiskit-Rustiq-greedy-preserve":3077,
                "Qiskit-Rustiq-greedy-preserve":2541,
              "QuCLEAR-transpiled": 2470,
              "MonteQ-greedy-whole": 2129,
                "MonteQ-greedy-sets-div": 2299,
                  "TKet-whole": 4738}

UCC_2_4_results ={"Qiskit-Rustiq-greedy-preserve":21,
                  "QuCLEAR": 12,
                  "MonteQ-greedy-fulldiv": 18,
                 "TKet": 23}
UCC_2_6_results ={"Qiskit-Rustiq-greedy-preserve":77,
                  "QuCLEAR":55,
                  "MonteQ-greedy-fulldiv": 71,
                 "TKet": 111}
UCC_4_8_results ={"Qiskit-Rustiq-greedy-preserve":330,
                  "QuCLEAR":264,
                  "MonteQ-greedy-fulldiv": 254,
                 "TKet": 514}
UCC_6_12_results ={"Qiskit-Rustiq-greedy-preserve":1923,
                   "QuCLEAR":1635,
                  "MonteQ-greedy-fulldiv": 1384,
                  "TKet": 3129}
UCC_8_16_results ={"Qiskit-Rustiq-greedy-preserve":7397,
                   "QuCLEAR":6099,
                  "MonteQ-greedy-fulldiv": 4849,
                  "TKet":11476}
UCC_10_20_results ={"Qiskit-Rustiq-greedy-preserve":21614,
                    "QuCLEAR":17877,
                  "MonteQ-greedy-fulldiv": 13065,
                   "Tket": 30871}

In [1]:
import numpy as np
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.transformers import FreezeCoreTransformer, ActiveSpaceTransformer

In [1]:
import pennylane as qml
available_data = qml.data.list_datasets()
print(available_data.keys()) # Shows top-level dataset categories, e.g., "qchem", "qspin"
print(available_data['qchem'].keys())

dict_keys(['qchem', 'qspin', 'other'])
dict_keys(['O2', 'H5', 'HeH+', 'N2H4', 'He2', 'CH2O', 'LiH', 'CH4', 'CO2', 'H2', 'H10', 'H2O', 'HCN', 'Li2', 'BH3', 'H6', 'C2H2', 'H4', 'H3+', 'N2H2', 'O3', 'OH-', 'C2', 'CH2', 'C2H6', 'NH3', 'CO', 'H2O2', 'NeH+', 'N2', 'H7', 'BeH2', 'H8', 'HF', 'C2H4'])


In [8]:
molecule = qml.data.load('qchem', molname='H2O')[0].molecule
#molecule_object = H2_data.molecule 
molecule_geo = [molecule.symbols, 0.529177*molecule.coordinates]
molecule_geo

[['O', 'H', 'H'],
 tensor([[ 0.11727153,  0.        ,  0.        ],
         [-0.46908611, -0.75696367,  0.        ],
         [-0.46908611,  0.75696367,  0.        ]], requires_grad=False)]

In [5]:
print(molecule)

<Molecule = CO, Charge: 0, Basis: STO-3G, Orbitals: 10, Electrons: 14>


In [2]:
theta = (180-107.6)/2*np.pi/180
dist = 1.0
H2 = "H 0 0 0; H 0 0 0.735"
CO = "C 0 0 0; O 0 0 1.12820186"
LiH = "Li 0.39247565 0 0; H -1.17742695 0 0"
H2O = 'O 0 0 0; H ' + str(dist*np.cos(theta)) + ' ' + str(-dist*np.sin(theta)) + ' 0; H ' + str(-dist*np.cos(theta)) + ' ' + str(-dist*np.sin(theta)) + ' 0'
BeH2 = "Be 0 0 0; H -1.3295022 0 0; H 1.3295022 0 0"
NH3 = "N 0 0 0.11401618; H 0 0.93857401 -0.26603774; H -0.81282894 -0.46928701 -0.26603774; H 0.81282894 -0.46928701 -0.26603774"
C2H4 = "C 0 0 0.66950111; C 0 0 -0.66950111; H 0 0.92890153 1.23210204; H 0 -0.92890153 1.23210204; H 0 0.92890153 -1.23210204; H 0 -0.92890153 -1.23210204"
C6H6 = "C -0.65914 -1.21034 3.98683;C 0.73798 -1.21034 4.02059;C -1.35771 -0.00006 3.96990;C 1.43653 -0.00004 4.03741;C -0.65915 1.21024 3.98685;C 0.73797 1.21024 4.02061;H -1.20447 -2.15520 3.97369;H 1.28332 -2.15517 4.03382;H -2.44839 -0.00006 3.94342;H 2.52722 -0.00004 4.06369;H -1.20448 2.15509 3.97373;H 1.28330 2.15508 4.03386;"

In [6]:
#H2 Hamiltonian
driver = PySCFDriver(
    atom=H2,
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

full_problem = driver.run()
fc_transformer = FreezeCoreTransformer()
fc_problem = fc_transformer.transform(full_problem)

fermionic_op = fc_problem.hamiltonian.second_q_op()
mapper = JordanWignerMapper()
qubit_op = mapper.map(fermionic_op)
print(qubit_op)


SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.81054798+0.j,  0.17218393+0.j, -0.22575349+0.j,  0.12091263+0.j,
  0.17218393+0.j,  0.16892754+0.j, -0.22575349+0.j,  0.16614543+0.j,
  0.0452328 +0.j,  0.0452328 +0.j,  0.0452328 +0.j,  0.0452328 +0.j,
  0.16614543+0.j,  0.17464343+0.j,  0.12091263+0.j])


In [7]:
#LiH Hamiltonian
driver = PySCFDriver(
    atom=LiH,
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

full_problem = driver.run()
fc_transformer = FreezeCoreTransformer()
fc_problem = fc_transformer.transform(full_problem)

fermionic_op = fc_problem.hamiltonian.second_q_op()
mapper = JordanWignerMapper()
qubit_op = mapper.map(fermionic_op)
print(qubit_op)


SparsePauliOp(['IIIIIIIIII', 'IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIIZZ', 'IIIIIYZZYI', 'IIIIIYZZYZ', 'IIIIIXZZXI', 'IIIIIXZZXZ', 'IIIIIIIZII', 'IIIIIIIZIZ', 'IIIIIIZIII', 'IIIIIIZIIZ', 'IIIIIZIIII', 'IIIIIZIIIZ', 'IIIIZIIIII', 'IIIIZIIIIZ', 'IIIYYIIIII', 'IIIYYIIIIZ', 'IIIXXIIIII', 'IIIXXIIIIZ', 'YZZZYIIIII', 'YZZZYIIIIZ', 'XZZZXIIIII', 'XZZZXIIIIZ', 'IIIZIIIIII', 'IIIZIIIIIZ', 'YZZYIIIIII', 'YZZYIIIIIZ', 'XZZXIIIIII', 'XZZXIIIIIZ', 'IIZIIIIIII', 'IIZIIIIIIZ', 'IZIIIIIIII', 'IZIIIIIIIZ', 'ZIIIIIIIII', 'ZIIIIIIIIZ', 'IIIIIYZZZY', 'IIIIIYZZIY', 'IIIIIXZZZX', 'IIIIIXZZIX', 'IIIIIIIIYY', 'IIIIIIIZYY', 'IIIIIIIIXX', 'IIIIIIIZXX', 'IIIIIIZIYY', 'IIIIIIZIXX', 'IIIIIZIIYY', 'IIIIIZIIXX', 'IIIIZIIIYY', 'IIIIZIIIXX', 'IIIYYIIIYY', 'IIIXXIIIYY', 'IIIYYIIIXX', 'IIIXXIIIXX', 'YZZZYIIIYY', 'XZZZXIIIYY', 'YZZZYIIIXX', 'XZZZXIIIXX', 'IIIZIIIIYY', 'IIIZIIIIXX', 'YZZYIIIIYY', 'XZZXIIIIYY', 'YZZYIIIIXX', 'XZZXIIIIXX', 'IIZIIIIIYY', 'IIZIIIIIXX', 'IZIIIIIIYY', 'IZIIIIIIXX', 'ZIIIIIIIYY', 'ZIIIIIIIXX', 'IIII

In [8]:
#H2O Hamiltonian
driver = PySCFDriver(
    atom=H2O,
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

full_problem = driver.run()
fc_transformer = FreezeCoreTransformer()
fc_problem = fc_transformer.transform(full_problem)

fermionic_op = fc_problem.hamiltonian.second_q_op()
mapper = JordanWignerMapper()
qubit_op = mapper.map(fermionic_op)
print(qubit_op)


SparsePauliOp(['IIIIIIIIIIII', 'IIIIIIIIIIIZ', 'IIIIIIIIIIZI', 'IIIIIIIIIIZZ', 'IIIIIIYZZZYI', 'IIIIIIYZZZYZ', 'IIIIIIXZZZXI', 'IIIIIIXZZZXZ', 'IIIIIIIIIZII', 'IIIIIIIIIZIZ', 'IIIIIIIYZYII', 'IIIIIIIYZYIZ', 'IIIIIIIXZXII', 'IIIIIIIXZXIZ', 'IIIIIIIIZIII', 'IIIIIIIIZIIZ', 'IIIIIIIZIIII', 'IIIIIIIZIIIZ', 'IIIIIIZIIIII', 'IIIIIIZIIIIZ', 'IIIIIZIIIIII', 'IIIIIZIIIIIZ', 'IIIYZYIIIIII', 'IIIYZYIIIIIZ', 'IIIXZXIIIIII', 'IIIXZXIIIIIZ', 'IYZZZYIIIIII', 'IYZZZYIIIIIZ', 'IXZZZXIIIIII', 'IXZZZXIIIIIZ', 'IIIIZIIIIIII', 'IIIIZIIIIIIZ', 'YZZZYIIIIIII', 'YZZZYIIIIIIZ', 'XZZZXIIIIIII', 'XZZZXIIIIIIZ', 'IIIZIIIIIIII', 'IIIZIIIIIIIZ', 'IYZYIIIIIIII', 'IYZYIIIIIIIZ', 'IXZXIIIIIIII', 'IXZXIIIIIIIZ', 'IIZIIIIIIIII', 'IIZIIIIIIIIZ', 'IZIIIIIIIIII', 'IZIIIIIIIIIZ', 'ZIIIIIIIIIII', 'ZIIIIIIIIIIZ', 'IIIIIIIIIYZY', 'IIIIIIIIIYIY', 'IIIIIIIIIXZX', 'IIIIIIIIIXIX', 'IIIIIIIYZZZY', 'IIIIIIIYZZIY', 'IIIIIIIXZZZX', 'IIIIIIIXZZIX', 'IIIIIIYZZYYY', 'IIIIIIXZZXYY', 'IIIIIIYZZXXY', 'IIIIIIXZZYYX', 'IIIIIIYZZYXX', 'IIIIIIXZ

In [9]:
#BeH2 Hamiltonian
driver = PySCFDriver(
    atom=BeH2,
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

full_problem = driver.run()
fc_transformer = FreezeCoreTransformer()
fc_problem = fc_transformer.transform(full_problem)

fermionic_op = fc_problem.hamiltonian.second_q_op()
mapper = JordanWignerMapper()
qubit_op = mapper.map(fermionic_op)
print(qubit_op)


SparsePauliOp(['IIIIIIIIIIII', 'IIIIIIIIIIIZ', 'IIIIIIIIIIZI', 'IIIIIIIIIIZZ', 'IIIIIIYZZZYI', 'IIIIIIYZZZYZ', 'IIIIIIXZZZXI', 'IIIIIIXZZZXZ', 'IIIIIIIIIZII', 'IIIIIIIIIZIZ', 'IIIIIIIIZIII', 'IIIIIIIIZIIZ', 'IIIIIIIZIIII', 'IIIIIIIZIIIZ', 'IIIIIIZIIIII', 'IIIIIIZIIIIZ', 'IIIIIZIIIIII', 'IIIIIZIIIIIZ', 'IYZZZYIIIIII', 'IYZZZYIIIIIZ', 'IXZZZXIIIIII', 'IXZZZXIIIIIZ', 'IIIIZIIIIIII', 'IIIIZIIIIIIZ', 'YZZZYIIIIIII', 'YZZZYIIIIIIZ', 'XZZZXIIIIIII', 'XZZZXIIIIIIZ', 'IIIZIIIIIIII', 'IIIZIIIIIIIZ', 'IIZIIIIIIIII', 'IIZIIIIIIIIZ', 'IZIIIIIIIIII', 'IZIIIIIIIIIZ', 'ZIIIIIIIIIII', 'ZIIIIIIIIIIZ', 'IIIIIIIYZZZY', 'IIIIIIIYZZIY', 'IIIIIIIXZZZX', 'IIIIIIIXZZIX', 'IIIIIIYYIIYY', 'IIIIIIXXIIYY', 'IIIIIIYXIIXY', 'IIIIIIXYIIYX', 'IIIIIIYYIIXX', 'IIIIIIXXIIXX', 'IIIIYYIIIIYY', 'IIIIXXIIIIYY', 'IIIIYYIIIIXX', 'IIIIXXIIIIXX', 'YZZZZYIIIIYY', 'XZZZZXIIIIYY', 'YZZZZYIIIIXX', 'XZZZZXIIIIXX', 'IYZZYIIIIIYY', 'IXZZXIIIIIYY', 'IYZZYIIIIIXX', 'IXZZXIIIIIXX', 'YYIIIIIIIIYY', 'XXIIIIIIIIYY', 'YYIIIIIIIIXX', 'XXIIIIII

In [10]:
#NH3 Hamiltonian
driver = PySCFDriver(
    atom=NH3,
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

full_problem = driver.run()
fc_transformer = FreezeCoreTransformer()
fc_problem = fc_transformer.transform(full_problem)

fermionic_op = fc_problem.hamiltonian.second_q_op()
mapper = JordanWignerMapper()
qubit_op = mapper.map(fermionic_op)
print(qubit_op)


SparsePauliOp(['IIIIIIIIIIIIII', 'IIIIIIIIIIIIIZ', 'IIIIIIIIIIIIZI', 'IIIIIIIIIIIIZZ', 'IIIIIIIYZZZZYI', 'IIIIIIIYZZZZYZ', 'IIIIIIIXZZZZXI', 'IIIIIIIXZZZZXZ', 'IIIIIIIIIIIZII', 'IIIIIIIIIIIZIZ', 'IIIIIIIIYZZYII', 'IIIIIIIIYZZYIZ', 'IIIIIIIIXZZXII', 'IIIIIIIIXZZXIZ', 'IIIIIIIIIIZIII', 'IIIIIIIIIIZIIZ', 'IIIIIIIIIYYIII', 'IIIIIIIIIYYIIZ', 'IIIIIIIIIXXIII', 'IIIIIIIIIXXIIZ', 'IIIIIIIIIZIIII', 'IIIIIIIIIZIIIZ', 'IIIIIIIIZIIIII', 'IIIIIIIIZIIIIZ', 'IIIIIIIZIIIIII', 'IIIIIIIZIIIIIZ', 'IIIIIIZIIIIIII', 'IIIIIIZIIIIIIZ', 'IIIYZZYIIIIIII', 'IIIYZZYIIIIIIZ', 'IIIXZZXIIIIIII', 'IIIXZZXIIIIIIZ', 'IIYZZZYIIIIIII', 'IIYZZZYIIIIIIZ', 'IIXZZZXIIIIIII', 'IIXZZZXIIIIIIZ', 'IIIIIZIIIIIIII', 'IIIIIZIIIIIIIZ', 'YZZZZYIIIIIIII', 'YZZZZYIIIIIIIZ', 'XZZZZXIIIIIIII', 'XZZZZXIIIIIIIZ', 'IIIIZIIIIIIIII', 'IIIIZIIIIIIIIZ', 'IYZZYIIIIIIIII', 'IYZZYIIIIIIIIZ', 'IXZZXIIIIIIIII', 'IXZZXIIIIIIIIZ', 'IIIZIIIIIIIIII', 'IIIZIIIIIIIIIZ', 'IIYYIIIIIIIIII', 'IIYYIIIIIIIIIZ', 'IIXXIIIIIIIIII', 'IIXXIIIIIIIIIZ', 'IIZIIIIIIIII

In [12]:
#C2H4 Hamiltonian
driver = PySCFDriver(
    atom=CO,
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

full_problem = driver.run()
fc_transformer = FreezeCoreTransformer()
fc_problem = fc_transformer.transform(full_problem)

fermionic_op = fc_problem.hamiltonian.second_q_op()
mapper = JordanWignerMapper()
qubit_op = mapper.map(fermionic_op)
print(qubit_op)


SparsePauliOp(['IIIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIZI', 'IIIIIIIIIIIIIIZZ', 'IIIIIIIIIIIYZZYI', 'IIIIIIIIIIIYZZYZ', 'IIIIIIIIIIIXZZXI', 'IIIIIIIIIIIXZZXZ', 'IIIIIIIIYZZZZZYI', 'IIIIIIIIYZZZZZYZ', 'IIIIIIIIXZZZZZXI', 'IIIIIIIIXZZZZZXZ', 'IIIIIIIIIIIIIZII', 'IIIIIIIIIIIIIZIZ', 'IIIIIIIIIIYZZYII', 'IIIIIIIIIIYZZYIZ', 'IIIIIIIIIIXZZXII', 'IIIIIIIIIIXZZXIZ', 'IIIIIIIIIYZZZYII', 'IIIIIIIIIYZZZYIZ', 'IIIIIIIIIXZZZXII', 'IIIIIIIIIXZZZXIZ', 'IIIIIIIIIIIIZIII', 'IIIIIIIIIIIIZIIZ', 'IIIIIIIIIIYZYIII', 'IIIIIIIIIIYZYIIZ', 'IIIIIIIIIIXZXIII', 'IIIIIIIIIIXZXIIZ', 'IIIIIIIIIYZZYIII', 'IIIIIIIIIYZZYIIZ', 'IIIIIIIIIXZZXIII', 'IIIIIIIIIXZZXIIZ', 'IIIIIIIIIIIZIIII', 'IIIIIIIIIIIZIIIZ', 'IIIIIIIIYZZYIIII', 'IIIIIIIIYZZYIIIZ', 'IIIIIIIIXZZXIIII', 'IIIIIIIIXZZXIIIZ', 'IIIIIIIIIIZIIIII', 'IIIIIIIIIIZIIIIZ', 'IIIIIIIIIZIIIIII', 'IIIIIIIIIZIIIIIZ', 'IIIIIIIIZIIIIIII', 'IIIIIIIIZIIIIIIZ', 'IIIIIIIZIIIIIIII', 'IIIIIIIZIIIIIIIZ', 'IIIIIIYYIIIIIIII', 'IIIIIIYYIIIIIIIZ', 'IIIIIIXXIIIIIIII', 'IIII